# Task termination: decide what a task measures

## Point

A task gives an agent several attempts, but its stopping rule determines which question you can answer. Stopping at first success measures discovery. A fixed episode budget also measures how well the agent reuses what it discovered.

## Objective

Use `terminate_task` for success and `max_task_episodes` as the attempt budget. Unlike the [commute example](00_episodes_to_tasks.ipynb), which continues after success, this example ends the task as soon as the agent reaches the goal.

`terminate_task` runs on every `step()`, including the reset frame. Returning true ends the task on that step. This callback returns true only when the episode terminates with a positive reward, so the task ends on the goal step. The episode budget truncates the task when an episode ends. The callback receives that step's reward, which indicates success in this FrozenLake example.

The environment is a deterministic 2×2 FrozenLake with a two-step episode limit and two episodes per task. The cell shows three outcomes:

1. The first episode reaches its step limit. The task continues with another attempt.
2. The second episode reaches the goal and the task ends.
3. A new task never reaches the goal, so it truncates when the second episode ends.

These actions are scripted to show the stopping rules; they do not demonstrate learning. FrozenLake's map is fixed here, so a new task does not draw a new map. The final observation of an episode and the reset observation that starts the next one arrive on separate steps. That reset step ignores its action and returns reward 0.


In [1]:
import gymnasium as gym

from repeat_task_gym import RepeatTaskEnv


def solved(*, reward, episode_terminated, **kwargs):
    return episode_terminated and reward > 0


base = gym.make(
    "FrozenLake-v1",
    desc=["SF", "FG"],
    is_slippery=False,
    max_episode_steps=2,
)
env = RepeatTaskEnv(base, max_task_episodes=2, terminate_task=solved)
try:
    env.reset(seed=0)
    env.step(0)  # Stay at the start.
    observation, _, terminated, truncated, _ = env.step(0)
    print("Inner timeout:", observation, terminated, truncated)
    assert observation["episode_truncated"] and not observation["episode_terminated"]
    assert not terminated and not truncated

    observation, reward, _, _, _ = env.step(0)  # Episode reset frame.
    assert not observation["episode_terminated"] and not observation["episode_truncated"]
    assert reward == 0.0
    env.step(2)  # Right.
    observation, reward, terminated, truncated, _ = env.step(1)  # Down.
    print("Solved task:", observation, reward, terminated, truncated)
    assert observation["episode_terminated"] and observation["episode_truncated"]
    assert terminated and truncated  # Success and the episode budget both apply.

    env.reset()
    for _ in range(2):
        env.step(0)
    env.step(0)  # Episode reset frame.
    env.step(0)
    observation, _, terminated, truncated, _ = env.step(0)
    print("Task timeout:", observation, terminated, truncated)
    assert not terminated and truncated
finally:
    env.close()


Inner timeout: {'observation': 0, 'episode_terminated': False, 'episode_truncated': True} False False
Solved task: {'observation': 3, 'episode_terminated': True, 'episode_truncated': True} 1.0 True True
Task timeout: {'observation': 0, 'episode_terminated': False, 'episode_truncated': True} False True


## Choose a stopping rule before comparing agents

The successful task took two attempts to reach the goal; the unsuccessful task exhausted the same budget. For first-success evaluation, report success rate and attempts to success, including which tasks exhausted the budget. To study performance after discovery, omit `terminate_task` and keep a fixed episode budget, as in notebook 00.

Keep the stopping rule and budget the same across agents. Otherwise a task's total reward or length can reflect different opportunities to act.

Next: [03 — RNG seeding control](03_rng_seeding_control.ipynb) shows how a plain `reset()` starts a new task and how optional seeds make a comparison reproducible.
